# Week 3 — Session 8
## Hypothesis Testing

### Goals

- Understand Null and Alternative Hypotheses
- Interpret p-values correctly
- Distinguish Reject from Fail to Reject
- Compare Statistical and Practical Significance
- Use One-Sample, Independent, and Paired t-tests
- Apply Hypothesis Testing to Tehran urban questions

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

# Part 1 — One-Sample t-test

Research Question:

Is the average walking time to metro greater than 10 minutes?

Null Hypothesis:

H0: μ = 10

Alternative Hypothesis:

H1: μ > 10

In [3]:
walking_time = np.array([
    11.2, 12.5, 10.8, 13.1, 9.9,
    11.7, 12.0, 10.6, 11.9, 12.3,
    13.0, 10.9, 11.4, 12.1, 11.6,
    10.7, 12.8, 11.3, 12.2, 11.8
])

walking_time

array([11.2, 12.5, 10.8, 13.1,  9.9, 11.7, 12. , 10.6, 11.9, 12.3, 13. ,
       10.9, 11.4, 12.1, 11.6, 10.7, 12.8, 11.3, 12.2, 11.8])

In [4]:
mean = walking_time.mean()
std = walking_time.std(ddof=1)
n = len(walking_time)
se = std / np.sqrt(n)

print("Mean:", mean)
print("Standard Deviation:", std)
print("Sample Size:", n)
print("Standard Error:", se)

Mean: 11.690000000000001
Standard Deviation: 0.8527972548588186
Sample Size: 20
Standard Error: 0.1906912632889531


In [5]:
result = stats.ttest_1samp(
    walking_time,
    popmean=10,
    alternative="greater"
)

print("Test Statistic:", result.statistic)
print("p-value:", result.pvalue)

Test Statistic: 8.862493073105066
p-value: 1.773164576362042e-08


In [6]:
alpha = 0.05

if result.pvalue < alpha:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Reject H0


## Interpretation

If p-value < 0.05:

We have sufficient statistical evidence against H0.

This does NOT mean:

"The probability that H0 is true is less than 5%."

It means that the observed data would be relatively unusual if H0 were true.

In [7]:
ci = stats.t.interval(
    confidence=0.95,
    df=n - 1,
    loc=mean,
    scale=se
)

print("95% Confidence Interval:", ci)

95% Confidence Interval: (np.float64(11.290878598970714), np.float64(12.089121401029288))


In [8]:
difference = mean - 10

print("Observed Difference:", difference)

Observed Difference: 1.6900000000000013


## Practical Interpretation

Statistical significance is not enough.

We also need to ask:

- How large is the difference?
- Is this difference important in practice?
- Is the sample representative?
- Is the uncertainty acceptable?

# Part 2 — Independent Samples t-test

Research Question:

Is average walking time to metro different between two neighborhoods?

Example:

Darrous vs Jordan

In [9]:
darrous = np.array([
    12.1, 11.7, 13.2, 10.9, 12.5,
    11.8, 12.9, 13.1, 11.5, 12.3
])

jordan = np.array([
    9.4, 10.1, 9.8, 10.5, 9.7,
    10.2, 9.9, 10.4, 9.6, 10.0
])

In [10]:
print("Darrous Mean:", darrous.mean())
print("Jordan Mean:", jordan.mean())

difference = darrous.mean() - jordan.mean()

print("Mean Difference:", difference)

Darrous Mean: 12.2
Jordan Mean: 9.959999999999999
Mean Difference: 2.24


In [11]:
result_ind = stats.ttest_ind(
    darrous,
    jordan,
    equal_var=False
)

print("Test Statistic:", result_ind.statistic)
print("p-value:", result_ind.pvalue)

Test Statistic: 8.601258708825517
p-value: 1.1279614142971316e-06


## Why Welch's t-test?

We use:

equal_var=False

This applies Welch's t-test.

It does not require us to assume that the two groups have exactly equal variances.

In [12]:
if result_ind.pvalue < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Reject H0


## Interpretation

If H0 is rejected:

We have evidence that the population means differ.

But we still need to evaluate:

- Mean Difference
- Confidence Interval
- Effect Size
- Sampling Quality
- Practical Importance

# Part 3 — Paired t-test

Research Question:

Did walking time decrease after an urban intervention?

The same units are measured:

Before → After

Therefore the data are Paired.

In [13]:
before = np.array([
    15.2, 14.8, 16.1, 15.5, 14.9,
    16.4, 15.7, 14.6, 15.9, 16.0
])

after = np.array([
    13.8, 13.5, 14.7, 14.1, 13.6,
    15.0, 14.2, 13.4, 14.5, 14.6
])

In [14]:
print("Before Mean:", before.mean())
print("After Mean:", after.mean())

Before Mean: 15.51
After Mean: 14.14


In [15]:
differences = after - before

print("Differences:")
print(differences)

print("Mean Difference:", differences.mean())

Differences:
[-1.4 -1.3 -1.4 -1.4 -1.3 -1.4 -1.5 -1.2 -1.4 -1.4]
Mean Difference: -1.37


In [16]:
result_paired = stats.ttest_rel(
    after,
    before,
    alternative="less"
)

print("Test Statistic:", result_paired.statistic)
print("p-value:", result_paired.pvalue)

Test Statistic: -52.623157652420225
p-value: 8.118648050775745e-13


In [17]:
if result_paired.pvalue < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Reject H0


## Interpretation

A Paired Test evaluates change within the same units.

This is different from comparing two unrelated groups.

The key variable becomes:

Difference = After - Before

# Type I and Type II Errors

Type I Error:

Detecting an effect that does not actually exist.

False Positive.

Type II Error:

Failing to detect a real effect.

False Negative.

Statistical Power:

The ability of a study to detect a real effect.

# Tehran Data-to-Design Intelligence

Hypothesis Testing can support several types of urban questions.

## Use Case 1 — Benchmark Test

Question:

Is average walking time to metro above an acceptable threshold?

Method:

One-Sample Test

## Use Case 2 — Neighborhood Comparison

Question:

Is accessibility significantly different between Darrous and Jordan?

Method:

Independent Samples Test

## Use Case 3 — Before / After Intervention

Question:

Did a design intervention improve use of public space?

Method:

Paired Test

# Tehran Project Reporting Rule

Do not report only:

p < 0.05

Instead report:

1. Estimate
2. Effect / Difference
3. Confidence Interval
4. p-value
5. Sample Size
6. Sampling Method
7. Possible Bias
8. Practical Meaning
9. Limitations

# Error Log

## E011 — p > 0.05

Incorrect:

"There is no effect."

Correct:

"There is insufficient evidence to reject H0."

---

## E012 — p-value Interpretation

Incorrect:

"p-value is the probability that H0 is true."

Correct:

p-value describes how unusual the observed result would be under H0.

---

## E013 — Statistical vs Practical Significance

A statistically significant result may still be too small to matter in practice.

# Session Summary

Hypothesis Testing helps evaluate whether observed differences are compatible with a null hypothesis.

Key rules:

- p < alpha → Reject H0
- p >= alpha → Fail to Reject H0
- Fail to Reject does not mean H0 is proven true
- p-value is not the probability that H0 is true
- Statistical Significance is not Practical Significance
- One-Sample Test compares a sample with a reference value
- Independent Test compares separate groups
- Paired Test compares linked observations
- Good interpretation requires Effect, Uncertainty, Evidence, and Context

# Independent Exercise

For each case, identify the correct test type.

### Case A

Average park distance vs a 500-meter standard.

### Case B

Average park distance in Darrous vs Jordan.

### Case C

The same 30 public spaces measured before and after redesign.

Then explain why each test type is appropriate.